# EEG-Based Neurological Disorder Detection

In this notebook, we will build a machine learning model that can classify EEG (Electroencephalogram) brain signals into 7 categories:

| Class | Condition | Key EEG Pattern |
|-------|-----------|------------------|
| 0 | Healthy | Normal alpha rhythm (8–13 Hz) |
| 1 | Interictal | Abnormal activity between seizures |
| 2 | Epilepsy | High-frequency spike bursts |
| 3 | Parkinson's | Slowed alpha, increased delta/theta |
| 4 | Alzheimer's | Very high delta, reduced alpha |
| 5 | ADHD | Elevated theta, reduced beta |
| 6 | Autism | Atypical gamma, elevated theta |

We will use the **Bonn University EEG Dataset** (Andrzejak et al., 2001) as our primary real-world data source, and generate synthetic data for disorders where real clinical data is not available.

## Step 1 — Import Libraries

In [1]:
import os
import numpy as np
from scipy import signal
from scipy.stats import skew, kurtosis
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully.")

All libraries imported successfully.


## Step 2 — Define Constants

The Bonn EEG dataset was recorded at a sampling rate of **173.61 Hz**. Each segment is **23.6 seconds** long, which gives us approximately **4097 data points** per segment.

We also define the standard clinical EEG frequency bands:
- **Delta (0.5–4 Hz):** Deep sleep, brain injury
- **Theta (4–8 Hz):** Drowsiness, ADHD marker
- **Alpha (8–13 Hz):** Relaxed wakefulness
- **Beta (13–30 Hz):** Active thinking, alertness
- **Gamma (30–60 Hz):** Cognitive processing

In [2]:
FS = 173.61
DURATION = 23.6
N_SAMPLES = 4097

BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 60),
}

CLASS_NAMES = ["Healthy", "Interictal", "Epilepsy", "Parkinsons", "Alzheimers", "ADHD", "Autism"]

print(f"Sampling Frequency : {FS} Hz")
print(f"Segment Duration   : {DURATION} s")
print(f"Samples/Segment    : {N_SAMPLES}")
print(f"Classes            : {CLASS_NAMES}")

Sampling Frequency : 173.61 Hz
Segment Duration   : 23.6 s
Samples/Segment    : 4097
Classes            : ['Healthy', 'Interictal', 'Epilepsy', 'Parkinsons', 'Alzheimers', 'ADHD', 'Autism']


## Step 3 — Load and Preprocess the Bonn EEG Dataset

The Bonn dataset has 5 folders:

| Folder | Label | Description |
|--------|-------|-------------|
| Z/ | 0 (Healthy) | Eyes open, healthy volunteers |
| O/ | 0 (Healthy) | Eyes closed, healthy volunteers |
| N/ | 1 (Interictal) | Seizure-free, hippocampal region |
| F/ | 1 (Interictal) | Seizure-free, epileptogenic zone |
| S/ | 2 (Epilepsy) | During seizure activity |

For each file, we:
1. Apply a **bandpass filter (0.5–40 Hz)** to remove irrelevant frequency components
2. Apply a **notch filter at 50 Hz** to remove power line interference
3. **Normalize** the signal (z-score) so all signals are on the same scale
4. Extract **10 clinically relevant features** from the cleaned signal

In [3]:
# Bonn folder -> label mapping
BONN_FOLDERS = {"Z": 0, "O": 0, "N": 1, "F": 1, "S": 2}

data_dir = "data"
all_X = []
all_y = []

print("Loading Bonn EEG dataset...\n")

for folder, label in BONN_FOLDERS.items():
    path = os.path.join(data_dir, folder)
    if not os.path.exists(path):
        print(f"  [SKIP] {path} not found")
        continue
    
    files = [f for f in os.listdir(path) if f.lower().endswith(".txt")]
    print(f"  Loading {len(files)} files from {folder}/ -> class {label} ({CLASS_NAMES[label]})")
    
    loaded = 0
    for fname in files:
        try:
            raw = np.loadtxt(os.path.join(path, fname))
            if len(raw) < 100:
                continue
            
            # --- Preprocessing ---
            # Bandpass filter: keep only 0.5-40 Hz
            nyq = FS / 2.0
            low_n = max(0.001, min(0.5 / nyq, 0.99))
            high_n = max(0.001, min(40.0 / nyq, 0.99))
            b, a = signal.butter(4, [low_n, high_n], btype='band')
            eeg = signal.filtfilt(b, a, raw)
            
            # Notch filter: remove 50 Hz power line noise
            notch_n = 50.0 / nyq
            if notch_n < 1.0:
                b, a = signal.iirnotch(notch_n, Q=30)
                eeg = signal.filtfilt(b, a, eeg)
            
            # Z-score normalization
            std = np.std(eeg)
            if std < 1e-10:
                eeg = eeg - np.mean(eeg)
            else:
                eeg = (eeg - np.mean(eeg)) / std
            
            # --- Feature Extraction (10 features) ---
            features = []
            
            # Compute band powers using Welch's method
            bp = {}
            for band, (fmin, fmax) in BANDS.items():
                freqs, psd = signal.welch(eeg, FS, nperseg=min(256, len(eeg)))
                idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                bp[band] = float(np.trapz(psd[idx], freqs[idx]))
            
            total_power = sum(bp.values()) + 1e-10
            
            # Features 1-5: Relative band powers
            features.append(bp['delta'] / total_power)
            features.append(bp['theta'] / total_power)
            features.append(bp['alpha'] / total_power)
            features.append(bp['beta']  / total_power)
            features.append(bp['gamma'] / total_power)
            
            # Feature 6: Theta/Alpha ratio (ADHD biomarker)
            features.append(bp['theta'] / (bp['alpha'] + 1e-10))
            
            # Feature 7: Delta/Alpha ratio (Alzheimer's biomarker)
            features.append(bp['delta'] / (bp['alpha'] + 1e-10))
            
            # Feature 8: Hjorth Mobility
            diff1 = np.diff(eeg)
            var0 = np.var(eeg)
            var1 = np.var(diff1)
            if var0 < 1e-10:
                features.append(0.0)
            else:
                features.append(float(np.sqrt(var1 / var0)))
            
            # Feature 9: Spectral Entropy
            _, psd_full = signal.welch(eeg, FS, nperseg=min(256, len(eeg)))
            psd_norm = psd_full / (psd_full.sum() + 1e-10)
            features.append(float(-np.sum(psd_norm * np.log2(psd_norm + 1e-10))))
            
            # Feature 10: Peak Frequency
            freqs_full, psd_full2 = signal.welch(eeg, FS, nperseg=min(256, len(eeg)))
            features.append(float(freqs_full[np.argmax(psd_full2)]))
            
            all_X.append(np.array(features))
            all_y.append(label)
            loaded += 1
            
        except Exception as e:
            print(f"    [SKIP] {fname}: {e}")
    
    print(f"    Loaded: {loaded} files")

print(f"\nTotal Bonn samples loaded: {len(all_X)}")

Loading Bonn EEG dataset...

  Loading 100 files from Z/ -> class 0 (Healthy)
    Loaded: 100 files
  Loading 100 files from O/ -> class 0 (Healthy)
    Loaded: 100 files
  Loading 100 files from N/ -> class 1 (Interictal)
    Loaded: 100 files
  Loading 100 files from F/ -> class 1 (Interictal)
    Loaded: 100 files
  Loading 100 files from S/ -> class 2 (Epilepsy)
    Loaded: 100 files

Total Bonn samples loaded: 500


## Step 5 — Generate Synthetic EEG Data

Real clinical EEG data for rare disorders is very hard to obtain. To ensure our model has enough training data for all 7 classes, we generate **synthetic EEG signals**.

We create signals by combining sinusoidal waves at disorder-specific frequencies with random noise. The frequency profiles are based on published medical research:

- **Healthy:** Strong alpha (8–13 Hz), moderate beta
- **Epilepsy:** High-frequency spikes (30–50 Hz)
- **Interictal:** Moderate theta and beta abnormalities
- **Parkinson's:** Alpha shifts down to 7–8 Hz, beta is suppressed
- **Alzheimer's:** Delta dominates, alpha is severely reduced
- **ADHD:** Theta is elevated, beta is reduced
- **Autism:** Unusual gamma activity, elevated theta

In [5]:
synthetic_per_class = 1000

# Count how many real samples we have for each class
real_counts = {i: 0 for i in range(7)}
for lbl in all_y:
    real_counts[lbl] = real_counts.get(lbl, 0) + 1

print("Real sample counts per class:")
for i, name in enumerate(CLASS_NAMES):
    print(f"  {name}: {real_counts[i]}")

# Frequency profiles for each disorder (based on clinical literature)
disorder_profiles = {
    "Healthy": {
        "freqs": [(8, 13), (13, 20), (18, 25)],
        "amps":  [(2.5, 3.5), (0.8, 1.2), (0.3, 0.6)],
        "noise": (0.2, 0.4),
        "label": 0
    },
    "Interictal": {
        "freqs": [(5, 8), (20, 30), (1, 4)],
        "amps":  [(1.5, 2.5), (1.0, 2.0), (0.8, 1.5)],
        "noise": (0.3, 0.6),
        "label": 1
    },
    "Epilepsy": {
        "freqs": [(30, 50), (20, 30), (2, 5)],
        "amps":  [(3.0, 5.0), (1.5, 2.5), (1.0, 2.0)],
        "noise": (0.4, 0.8),
        "label": 2
    },
    "Parkinsons": {
        "freqs": [(4, 8), (0.5, 3), (4, 7), (13, 18)],
        "amps":  [(2.0, 3.0), (1.5, 2.5), (1.5, 2.5), (0.3, 0.6)],
        "noise": (0.3, 0.5),
        "label": 3
    },
    "Alzheimers": {
        "freqs": [(0.5, 2), (4, 7), (6, 9), (1, 3)],
        "amps":  [(3.0, 5.0), (2.0, 3.0), (0.5, 1.0), (1.5, 2.5)],
        "noise": (0.2, 0.4),
        "label": 4
    },
    "ADHD": {
        "freqs": [(4, 8), (4, 7), (8, 10), (13, 16)],
        "amps":  [(3.0, 4.5), (2.0, 3.0), (1.0, 1.8), (0.4, 0.7)],
        "noise": (0.3, 0.5),
        "label": 5
    },
    "Autism": {
        "freqs": [(30, 45), (4, 8), (8, 11), (35, 55)],
        "amps":  [(1.5, 3.0), (2.0, 3.5), (1.0, 1.8), (0.8, 1.8)],
        "noise": (0.3, 0.5),
        "label": 6
    },
}

print("\nGenerating synthetic data...\n")

for disorder_name, profile in disorder_profiles.items():
    label = profile["label"]
    n_synth = max(0, synthetic_per_class - real_counts[label])
    
    if n_synth == 0:
        print(f"  Skipping {disorder_name} — already have {real_counts[label]} real samples")
        continue
    
    print(f"  Generating {n_synth} samples for {disorder_name}...")
    
    for _ in range(n_synth):
        # Build the synthetic signal by adding sinusoidal components
        t = np.linspace(0, DURATION, N_SAMPLES)
        eeg_raw = np.zeros(N_SAMPLES)
        
        for freq_range, amp_range in zip(profile["freqs"], profile["amps"]):
            freq = np.random.uniform(freq_range[0], freq_range[1])
            amp = np.random.uniform(amp_range[0], amp_range[1])
            phase = np.random.uniform(0, 2 * np.pi)
            eeg_raw += amp * np.sin(2 * np.pi * freq * t + phase)
        
        # Add random noise
        noise_level = np.random.uniform(profile["noise"][0], profile["noise"][1])
        eeg_raw += noise_level * np.random.randn(N_SAMPLES)
        
        # --- Preprocessing ---
        nyq = FS / 2.0
        low_n = max(0.001, min(0.5 / nyq, 0.99))
        high_n = max(0.001, min(40.0 / nyq, 0.99))
        b, a = signal.butter(4, [low_n, high_n], btype='band')
        eeg = signal.filtfilt(b, a, eeg_raw)
        
        notch_n = 50.0 / nyq
        if notch_n < 1.0:
            b, a = signal.iirnotch(notch_n, Q=30)
            eeg = signal.filtfilt(b, a, eeg)
        
        std = np.std(eeg)
        if std < 1e-10:
            eeg = eeg - np.mean(eeg)
        else:
            eeg = (eeg - np.mean(eeg)) / std
        
        # --- Feature Extraction ---
        features = []
        bp = {}
        for band, (fmin, fmax) in BANDS.items():
            freqs, psd = signal.welch(eeg, FS, nperseg=min(256, len(eeg)))
            idx = np.logical_and(freqs >= fmin, freqs <= fmax)
            bp[band] = float(np.trapz(psd[idx], freqs[idx]))
        
        total_power = sum(bp.values()) + 1e-10
        features.append(bp['delta'] / total_power)
        features.append(bp['theta'] / total_power)
        features.append(bp['alpha'] / total_power)
        features.append(bp['beta']  / total_power)
        features.append(bp['gamma'] / total_power)
        features.append(bp['theta'] / (bp['alpha'] + 1e-10))
        features.append(bp['delta'] / (bp['alpha'] + 1e-10))
        
        diff1 = np.diff(eeg)
        var0, var1 = np.var(eeg), np.var(diff1)
        features.append(0.0 if var0 < 1e-10 else float(np.sqrt(var1 / var0)))
        
        _, psd_full = signal.welch(eeg, FS, nperseg=min(256, len(eeg)))
        psd_norm = psd_full / (psd_full.sum() + 1e-10)
        features.append(float(-np.sum(psd_norm * np.log2(psd_norm + 1e-10))))
        
        freqs_full, psd_full2 = signal.welch(eeg, FS, nperseg=min(256, len(eeg)))
        features.append(float(freqs_full[np.argmax(psd_full2)]))
        
        all_X.append(np.array(features))
        all_y.append(label)

print("\nSynthetic data generation complete.")
print(f"Total dataset size: {len(all_X)} samples")

Real sample counts per class:
  Healthy: 200
  Interictal: 200
  Epilepsy: 100
  Parkinsons: 0
  Alzheimers: 0
  ADHD: 0
  Autism: 0

Generating synthetic data...

  Generating 800 samples for Healthy...
  Generating 800 samples for Interictal...
  Generating 900 samples for Epilepsy...
  Generating 1000 samples for Parkinsons...
  Generating 1000 samples for Alzheimers...
  Generating 1000 samples for ADHD...
  Generating 1000 samples for Autism...

Synthetic data generation complete.
Total dataset size: 7000 samples


## Step 6 — Dataset Summary

Let us look at how many samples we have in each class after combining real + synthetic data.

In [6]:
X = np.array(all_X)
y = np.array(all_y)

print(f"Total samples   : {len(X)}")
print(f"Features/sample : {X.shape[1]}")
print()

for i, name in enumerate(CLASS_NAMES):
    count = (y == i).sum()
    src = "real+synth" if real_counts.get(i, 0) > 0 else "synthetic"
    print(f"  Class {i} ({name:12s}): {count:4d} samples  [{src}]")

Total samples   : 7000
Features/sample : 10

  Class 0 (Healthy     ): 1000 samples  [real+synth]
  Class 1 (Interictal  ): 1000 samples  [real+synth]
  Class 2 (Epilepsy    ): 1000 samples  [real+synth]
  Class 3 (Parkinsons  ): 1000 samples  [synthetic]
  Class 4 (Alzheimers  ): 1000 samples  [synthetic]
  Class 5 (ADHD        ): 1000 samples  [synthetic]
  Class 6 (Autism      ): 1000 samples  [synthetic]


## Step 7 — Train-Test Split and Feature Scaling

We split the data into 80% training and 20% testing. Stratified sampling ensures each class is proportionally represented in both sets.

**StandardScaler** normalizes each feature to zero mean and unit variance. We fit the scaler on training data only to prevent data leakage.

We also compute **balanced class weights** so the model gives equal importance to all classes.

In [7]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set : {len(X_tr)} samples")
print(f"Testing set  : {len(X_te)} samples")

# Scale features
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

# Compute class weights for balanced training
cw = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
cw_dict = dict(zip(np.unique(y_tr), cw))
print(f"\nClass weights: {cw_dict}")

Training set : 5600 samples
Testing set  : 1400 samples

Class weights: {np.int64(0): np.float64(1.0), np.int64(1): np.float64(1.0), np.int64(2): np.float64(1.0), np.int64(3): np.float64(1.0), np.int64(4): np.float64(1.0), np.int64(5): np.float64(1.0), np.int64(6): np.float64(1.0)}


## Step 8 — Train the Random Forest Model

We use a **Random Forest Classifier** with 300 trees. Random Forest is an ensemble method that trains multiple decision trees and averages their results. It works well for EEG classification because:
- It handles non-linear patterns in brain signals
- It is robust against noisy data
- It provides feature importance rankings
- It does not require heavy hyperparameter tuning

In [8]:
print("Training Random Forest (300 trees)...")

model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=1,
    class_weight=cw_dict,
    random_state=42,
    n_jobs=-1,
)

model.fit(X_tr_s, y_tr)
print("Training complete!")

Training Random Forest (300 trees)...
Training complete!


## Step 9 — Evaluate the Model

We check how well the model performs on the test set using:
- **Accuracy** — overall percentage of correct predictions
- **Weighted F1 Score** — balances precision and recall, weighted by class size
- **Macro F1 Score** — average F1 across all classes (treats each class equally)

In [9]:
y_pred = model.predict(X_te_s)

acc = accuracy_score(y_te, y_pred)
f1w = f1_score(y_te, y_pred, average='weighted')
f1m = f1_score(y_te, y_pred, average='macro')

print("-" * 52)
print(f"  Test Accuracy      : {acc * 100:.2f}%")
print(f"  Weighted F1 Score  : {f1w:.4f}")
print(f"  Macro F1 Score     : {f1m:.4f}")
print("-" * 52)

----------------------------------------------------
  Test Accuracy      : 99.07%
  Weighted F1 Score  : 0.9907
  Macro F1 Score     : 0.9907
----------------------------------------------------


### Per-Class Classification Report

This shows **precision**, **recall**, and **F1 score** for each individual class.

In [10]:
print(classification_report(y_te, y_pred, target_names=CLASS_NAMES))

              precision    recall  f1-score   support

     Healthy       0.98      0.98      0.98       200
  Interictal       0.98      0.99      0.99       200
    Epilepsy       0.99      0.97      0.98       200
  Parkinsons       0.99      0.99      0.99       200
  Alzheimers       1.00      0.99      1.00       200
        ADHD       0.99      1.00      0.99       200
      Autism       1.00      1.00      1.00       200

    accuracy                           0.99      1400
   macro avg       0.99      0.99      0.99      1400
weighted avg       0.99      0.99      0.99      1400



### Confusion Matrix

The confusion matrix tells us which classes the model confuses with each other. Values on the diagonal are correct predictions.

In [11]:
cm = confusion_matrix(y_te, y_pred)

header = "          " + "".join(f"{n[:8]:>10}" for n in CLASS_NAMES)
print(header)
for i, row in enumerate(cm):
    print(f"  {CLASS_NAMES[i][:10]:10s}" + "".join(f"{v:>10}" for v in row))

             Healthy  Interict  Epilepsy  Parkinso  Alzheime      ADHD    Autism
  Healthy          197         2         1         0         0         0         0
  Interictal         0       199         0         1         0         0         0
  Epilepsy           3         2       194         0         0         1         0
  Parkinsons         0         0         0       198         0         2         0
  Alzheimers         0         0         0         1       199         0         0
  ADHD               0         0         0         0         0       200         0
  Autism             0         0         0         0         0         0       200


## Step 10 — 5-Fold Cross-Validation

Cross-validation gives us a more reliable estimate of model performance. It splits data into 5 parts, trains on 4, tests on 1, and repeats 5 times. A low standard deviation means the model is consistent.

In [12]:
print("Running 5-fold cross-validation...\n")

X_all_s = scaler.transform(X)
cv = cross_val_score(model, X_all_s, y, cv=5, scoring='f1_weighted', n_jobs=-1)

print(f"  CV F1 scores : {[f'{s:.4f}' for s in cv]}")
print(f"  CV Mean F1   : {cv.mean():.4f} +/- {cv.std():.4f}")

Running 5-fold cross-validation...

  CV F1 scores : ['0.6631', '0.9979', '0.9964', '0.9964', '1.0000']
  CV Mean F1   : 0.9308 +/- 0.1338


## Step 11 — Feature Importance

Random Forest tells us which features contributed most to the classification. This helps us understand which EEG biomarkers are the strongest indicators of neurological disorders.

In [13]:
feat_names = [
    'delta_rel', 'theta_rel', 'alpha_rel', 'beta_rel', 'gamma_rel',
    'theta_alpha_ratio', 'delta_alpha_ratio',
    'hjorth_mobility', 'spectral_entropy', 'peak_frequency'
]

importances = model.feature_importances_
top5 = np.argsort(importances)[::-1][:5]

print("Top 5 Most Important Features:\n")
for i, idx in enumerate(top5):
    name = feat_names[idx] if idx < len(feat_names) else f'feature_{idx}'
    print(f"  {i+1}. {name:30s}: {importances[idx]:.4f}")

Top 5 Most Important Features:

  1. hjorth_mobility               : 0.1864
  2. beta_rel                      : 0.1768
  3. gamma_rel                     : 0.1330
  4. delta_alpha_ratio             : 0.1248
  5. theta_alpha_ratio             : 0.0821


## Step 12 — Save the Trained Model

We save the model and supporting files so they can be loaded by the API server for real-time predictions without retraining.

In [14]:
os.makedirs("model", exist_ok=True)

joblib.dump(model,       "model/eeg_model.pkl")
joblib.dump(scaler,      "model/scaler.pkl")
joblib.dump(CLASS_NAMES, "model/class_names.pkl")
joblib.dump(feat_names,  "model/feature_names.pkl")

print("-" * 52)
print(f"  Model saved to   : model/eeg_model.pkl")
print(f"  Classes           : {', '.join(CLASS_NAMES)}")
print(f"  Weighted F1       : {f1w:.4f}")
print(f"  Test Accuracy     : {acc * 100:.1f}%")
print("-" * 52)
print("\nDone! The model is ready for predictions.")

----------------------------------------------------
  Model saved to   : model/eeg_model.pkl
  Classes           : Healthy, Interictal, Epilepsy, Parkinsons, Alzheimers, ADHD, Autism
  Weighted F1       : 0.9907
  Test Accuracy     : 99.1%
----------------------------------------------------

Done! The model is ready for predictions.
